In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

PROJECT_ROOT = Path.cwd().parent
SPLITS_ROOT = PROJECT_ROOT / "data" / "splits"

CRIC_SPLIT = SPLITS_ROOT / "cric_splits.csv"
RIVA_SPLIT = SPLITS_ROOT / "riva_splits.csv"

cric_df = pd.read_csv(CRIC_SPLIT)
riva_df = pd.read_csv(RIVA_SPLIT)

RANDOM_SEED = 42

print("PHASE 8 — FIXED SPLITS & REPRODUCIBILITY")
print("=" * 55)

print("\nCRIC")
print(f"Learning units: {len(cric_df):,}")
print(f"Images: {cric_df['image_filename'].nunique():,}")
print(cric_df["split"].value_counts().sort_index())

print("\nRIVA")
print(f"Learning units: {len(riva_df):,}")
print(f"Images: {riva_df['image_filename'].nunique():,}")
print(f"Slides: {riva_df['slide_id'].nunique():,}")
print(riva_df["split"].value_counts().sort_index())

# Basic integrity checks
assert set(cric_df["split"].unique()) == {"train", "val", "test"}
assert set(riva_df["split"].unique()) == {"train", "val", "test"}

assert cric_df["split"].notna().all()
assert riva_df["split"].notna().all()

assert cric_df["image_filename"].notna().all()
assert riva_df["image_filename"].notna().all()

# CRIC image leakage
cric_image_splits = cric_df.groupby("image_filename")["split"].nunique()
assert cric_image_splits.max() == 1

# RIVA slide leakage
riva_slide_splits = riva_df.groupby("slide_id")["split"].nunique()
assert riva_slide_splits.max() == 1

print("\nIntegrity checks: PASSED")
print("CRIC image leakage: PASSED")
print("RIVA slide leakage: PASSED")
print(f"\nFixed random seed: {RANDOM_SEED}")

PHASE 8 — FIXED SPLITS & REPRODUCIBILITY

CRIC
Learning units: 11,534
Images: 400
split
test     1721
train    8274
val      1539
Name: count, dtype: int64

RIVA
Learning units: 15,949
Images: 959
Slides: 111
split
test      2552
train    10757
val       2640
Name: count, dtype: int64

Integrity checks: PASSED
CRIC image leakage: PASSED
RIVA slide leakage: PASSED

Fixed random seed: 42


In [2]:
# PHASE 8 — SAVE REPRODUCIBILITY CONFIGURATION

CONFIG_ROOT = PROJECT_ROOT / "configs"
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)

config = {
    "project": "cervical-semi-sl",
    "stage": "08_fixed_splits_reproducibility",
    "random_seed": 42,

    "datasets": {
        "CRIC": {
            "learning_units": int(len(cric_df)),
            "parent_unit": "image",
            "images": int(cric_df["image_filename"].nunique()),
            "split_file": "data/splits/cric_splits.csv"
        },
        "RIVA": {
            "learning_units": int(len(riva_df)),
            "parent_unit": "slide",
            "images": int(riva_df["image_filename"].nunique()),
            "slides_represented": int(riva_df["slide_id"].nunique()),
            "split_file": "data/splits/riva_splits.csv"
        }
    },

    "split_policy": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
        "cric_grouping": "image",
        "riva_grouping": "slide"
    },

    "label_definition": {
        "0": "Normal / non-pathological",
        "1": "Abnormal / pathological"
    },

    "status": "fixed"
}

CONFIG_FILE = CONFIG_ROOT / "phase8_reproducibility.json"

with open(CONFIG_FILE, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Reproducibility configuration saved:")
print(CONFIG_FILE)

Reproducibility configuration saved:
c:\Users\nanda\Documents\cervical-semi-sl\configs\phase8_reproducibility.json
